# DLBCL Cell Analysis Pipeline

This notebook is the clean interactive entry point for the repository. It keeps configuration in one place, previews the data layout, optionally runs the expensive Cellpose/ImageJ workflow, combines CSV outputs, and produces quick QC figures.

Run sections top to bottom. Processing cells are guarded by `RUN_PROCESSING = False` so opening the notebook does not accidentally rerun segmentation.

## 1. Setup

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_DIR = Path.cwd()
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from csvOps.combine_images import combine_sample
import csvOps.combine_patients as combine_patients
from produce_figures.plotResponderComparison import plot_all_comparison, plot_combined_comparison
from utils.channel_aliases import canonicalize_channel_config
from utils.config_helpers import extract_sample_number, filter_image_folders, normalize_image_filter_config

pd.set_option("display.max_columns", 120)
PROJECT_DIR

## 2. Configure Data and Channels

In [ ]:
BASE_DIR = PROJECT_DIR.parent
RESPONSE_GROUP = "non-responder"  # "responder" or "non-responder"
PATIENT_FOLDER = "01-03-2026 DLBCL 109241"
BASE_PATH = BASE_DIR / RESPONSE_GROUP / PATIENT_FOLDER

# Use [] or None for all samples. Use [1, 2, 3] for selected samples.
SAMPLES_TO_PROCESS = [1]

# Use {} for all images, {5} for the same image in every sample, or {1: [5, 13]} per sample.
IMAGES_TO_PROCESS = {5}

CHANNEL_CONFIG = canonicalize_channel_config({
    "actin": "processed_Actin-FITC.tif",
    "cd4": "processed_CD4-PerCP.tif",
    "cd45ra_PacBlue": "processed_CD45RA-PacBlue.tif",
    "cd19car": "processed_CD19CAR-AF647.tif",
    "ccr7": "processed_CCR7-AF594.tif",
})

BASE_PATH

In [ ]:
PARAMS = {
    "cellpose_model": "cyto2",
    "cellpose_diameter": 250,
    "cellpose_flow_threshold": 0.6,
    "cellpose_cellprob_threshold": -6.0,
    "cellpose_use_gpu": True,
    "min_size": 180,
    "max_size": 100000,
}

PARAMS

## 3. Inspect Patient Folder

In [ ]:
def list_sample_folders(base_path: Path, samples_to_process=None):
    sample_folders = [p.name for p in base_path.iterdir() if p.is_dir() and p.name.lower().startswith("sample")]
    sample_folders = sorted(sample_folders, key=extract_sample_number)
    if samples_to_process:
        sample_folders = [s for s in sample_folders if extract_sample_number(s) in set(samples_to_process)]
    return sample_folders

def list_image_folders(base_path: Path, sample_folder: str, image_filters):
    sample_path = base_path / sample_folder
    image_folders = [p.name for p in sample_path.iterdir() if p.is_dir()]
    image_folders = sorted(image_folders, key=lambda x: (not x.isdigit(), int(x) if x.isdigit() else x))
    filters_map, filters_default = normalize_image_filter_config(image_filters)
    return filter_image_folders(sample_folder, image_folders, filters_map, filters_default, announce=False)

sample_folders = list_sample_folders(BASE_PATH, SAMPLES_TO_PROCESS)
preview_rows = []
for sample in sample_folders:
    images = list_image_folders(BASE_PATH, sample, IMAGES_TO_PROCESS)
    preview_rows.append({"sample": sample, "images_selected": len(images), "images": ", ".join(images[:12])})

pd.DataFrame(preview_rows)

In [ ]:
def preview_channel_files(base_path: Path, sample_folders, image_filters, channel_config):
    rows = []
    for sample in sample_folders:
        for image in list_image_folders(base_path, sample, image_filters):
            image_dir = base_path / sample / image
            row = {"sample": sample, "image": image}
            for channel, filename in channel_config.items():
                row[channel] = (image_dir / filename).exists()
            rows.append(row)
    return pd.DataFrame(rows)

preview_channel_files(BASE_PATH, sample_folders, IMAGES_TO_PROCESS, CHANNEL_CONFIG)

## 4. Run Processing

This section initializes ImageJ and runs Cellpose. Change `RUN_PROCESSING` to `True` only when the configuration preview above looks correct.

In [ ]:
RUN_PROCESSING = False

processing_results = []
if RUN_PROCESSING:
    from pipeline_helpers import prepare_run, prompt_channel_filenames
    from process_single_image import run_pipeline

    base_path_obj, discovered_samples, ij = prepare_run(str(BASE_PATH), SAMPLES_TO_PROCESS)
    for sample in discovered_samples:
        for image in list_image_folders(BASE_PATH, sample, IMAGES_TO_PROCESS):
            channel_config_for_image = prompt_channel_filenames(base_path_obj, sample, image, CHANNEL_CONFIG)
            result = run_pipeline(
                sample_folder=sample,
                image_number=image,
                base_path=str(BASE_PATH),
                segmentation_method="cellpose",
                params=PARAMS,
                channel_config=channel_config_for_image,
                combine_channels=None,
                null_channels=None,
                ij=ij,
                verbose=True,
            )
            processing_results.append({"sample": sample, "image": image, "success": result["success"], "error": result.get("error")})
        combine_sample(sample_name=sample, base_path=str(BASE_PATH), verbose=True)

pd.DataFrame(processing_results)

## 5. Combine CSV Outputs

In [ ]:
sample_paths = [BASE_PATH / sample for sample in list_sample_folders(BASE_PATH, SAMPLES_TO_PROCESS)]
for sample_path in sample_paths:
    combine_sample(sample_name=sample_path.name, base_path=str(BASE_PATH), verbose=True)

sample_tables = []
for sample_path in sample_paths:
    sample_csv = sample_path / "combined_measurements.csv"
    if sample_csv.exists():
        sample_tables.append(pd.read_csv(sample_csv))

if not sample_tables:
    raise FileNotFoundError("No sample-level combined_measurements.csv files found.")

master_df = pd.concat(sample_tables, ignore_index=True)
if "global_cell_id" not in master_df.columns:
    master_df.insert(0, "global_cell_id", range(1, len(master_df) + 1))

master_csv = BASE_PATH / "all_samples_combined.csv"
master_df.to_csv(master_csv, index=False)
master_result = {"output_path": str(master_csv), "num_samples": len(sample_tables), "num_cells": len(master_df)}
master_result

In [ ]:
master_csv = BASE_PATH / "all_samples_combined.csv"
df = pd.read_csv(master_csv)
print(df.shape)
df.head()

## 6. Quick QC Plots

In [ ]:
qc_columns = [c for c in ["area", "cd4_mean", "ccr7_mean", "cd45ra_sparkviolet_mean", "cd19car_mean"] if c in df.columns]
axes = df[qc_columns].hist(bins=40, figsize=(14, 8), layout=(2, 3))
plt.tight_layout()

In [ ]:
if {"sample", "image"}.issubset(df.columns):
    display(df.groupby(["sample", "image"]).size().rename("cell_count").reset_index())

## 7. Combine Patients and Compare Response Groups

In [ ]:
RUN_GROUP_COMBINE = False

if RUN_GROUP_COMBINE:
    combine_patients.BASE_DIR = BASE_DIR
    combine_patients.combine_patient_csvs("responder")
    combine_patients.combine_patient_csvs("non-responder")
    plot_all_comparison(BASE_DIR, verbose=True)
    plot_combined_comparison(BASE_DIR, verbose=True)

## Notes

- Keep raw patient data outside the code repository root when possible.
- Commit code, notebooks, and small reference CSVs. Avoid committing generated TIFF crops, Cellpose outputs, and combined CSVs unless they are deliberate release artifacts.
- The older notebooks are preserved for reference, but this notebook should be the working entry point.